# Building Permits Survey: Place-Level Monthly Time Series

This notebook downloads and processes monthly building permit data from the Census Bureau's Building Permits Survey (BPS) at the **place level** (individual permit-issuing jurisdictions: cities, towns, MCDs).

**Data source:** https://www2.census.gov/econ/bps/Place/

**Coverage:** January 2000 – October 2025 (310 months, ~15,000–20,000 places)

**Four regions:** Northeast, Midwest, South, West (1,240 files total)

**Key challenge:** The CSV column structure changed four times over the study period. All formats share the same last 12 columns (permit data: bldgs/units/value × 4 structure types) but differ in metadata columns.

**Output:** Panel dataset of place × year-month with permit counts by structure type.

In [1]:
import pandas as pd
import numpy as np
import urllib.request
import time
from pathlib import Path

PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / 'output'
RAW_PLACE = PROJECT_DIR / 'raw_place'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_PLACE.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# ── URL configuration ──
BASE_URL = 'https://www2.census.gov/econ/bps/Place'
REGIONS = {
    'Northeast Region': 'ne',
    'Midwest Region': 'mw',
    'South Region': 'so',
    'West Region': 'we',
}

# ── Shared permit columns (last 12 in every format) ──
PERMIT_COLS = [
    'units_1_bldgs', 'units_1', 'units_1_value',
    'units_2_bldgs', 'units_2', 'units_2_value',
    'units_34_bldgs', 'units_34', 'units_34_value',
    'units_5plus_bldgs', 'units_5plus', 'units_5plus_value',
]

# ── Column definitions for each format era ──
# Format A (2000-2003): 26 cols — MSA/CMSA codes, no CBSA
COLS_A = [
    'survey_date', 'state_code', 'id_6digit', 'county_code', 'place_code',
    'msa_cmsa', 'pmsa_code', 'ccity', 'zip_code',
    'region', 'division', 'source', '_empty', 'place_name',
] + PERMIT_COLS

# Format B (2004): 26 cols — CSA/CBSA replaces MSA/CMSA
COLS_B = [
    'survey_date', 'state_code', 'id_6digit', 'county_code', 'place_code',
    'csa_code', 'cbsa_code', 'ccity', 'zip_code',
    'region', 'division', 'source', '_empty', 'place_name',
] + PERMIT_COLS

# Format B2 (Jan-Mar 2005): 27 cols — Footnote added, empty field before place_name
COLS_B2 = [
    'survey_date', 'state_code', 'id_6digit', 'county_code', 'place_code',
    'csa_code', 'cbsa_code', 'footnote', 'ccity', 'zip_code',
    'region', 'division', 'source', '_empty', 'place_name',
] + PERMIT_COLS

# Format B2b (Apr 2005 - Dec 2007): 26 cols — Footnote added, no empty field
COLS_B2b = [
    'survey_date', 'state_code', 'id_6digit', 'county_code', 'place_code',
    'csa_code', 'cbsa_code', 'footnote', 'ccity', 'zip_code',
    'region', 'division', 'source', 'place_name',
] + PERMIT_COLS

# Format C (2008-2025): 29 cols — FIPS Place, FIPS MCD, Pop added
COLS_C = [
    'survey_date', 'state_code', 'id_6digit', 'county_code', 'census_place',
    'fips_place', 'fips_mcd', 'pop', 'csa_code', 'cbsa_code',
    'footnote', 'central_city', 'zip_code',
    'region', 'division', 'source', 'place_name',
] + PERMIT_COLS

# Common output columns (after standardization)
COMMON_COLS = [
    'survey_date', 'state_code', 'id_6digit', 'county_code',
    'place_name', 'zip_code', 'region', 'division', 'source',
    'csa_code', 'cbsa_code', 'pop',
] + PERMIT_COLS

print(f"Format A: {len(COLS_A)} cols")
print(f"Format B: {len(COLS_B)} cols")
print(f"Format B2: {len(COLS_B2)} cols")
print(f"Format B2b: {len(COLS_B2b)} cols")
print(f"Format C: {len(COLS_C)} cols")
print(f"Common output: {len(COMMON_COLS)} cols")

Format A: 26 cols
Format B: 26 cols
Format B2: 27 cols
Format B2b: 26 cols
Format C: 29 cols
Common output: 24 cols


## 1. Download all place-level files

In [11]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# Generate manifest of all 1,240 files (4 regions × 310 months)
manifest = []
for region_dir, prefix in REGIONS.items():
    for year in range(2000, 2026):
        end_month = 10 if year == 2025 else 12
        for month in range(1, end_month + 1):
            yymm = f"{year % 100:02d}{month:02d}"
            fname = f"{prefix}{yymm}c.txt"
            url = f"{BASE_URL}/{region_dir.replace(' ', '%20')}/{fname}"
            manifest.append((url, fname))

print(f"Total files to download: {len(manifest)}")

# Filter to files not yet downloaded
to_download = [(url, fname) for url, fname in manifest
               if not (RAW_PLACE / fname).exists() or (RAW_PLACE / fname).stat().st_size < 100]
skipped = len(manifest) - len(to_download)
print(f"Already downloaded: {skipped}, remaining: {len(to_download)}")


def download_one(args, max_retries=5):
    url, fname = args
    dest = RAW_PLACE / fname

    for attempt in range(max_retries):
        try:
            urllib.request.urlretrieve(url, dest)
            if dest.exists() and dest.stat().st_size > 100:
                return fname, True, None
            else:
                raise Exception("Downloaded file too small")
        except Exception as e:
            err = str(e)
            wait = 0.5 * (attempt + 1)

            if '429' in err or 'SSL' in err or 'timed out' in err:
                time.sleep(wait)
            else:
                return fname, False, err

# Download with 20 concurrent threads
failures = []
downloaded = 0
with ThreadPoolExecutor(max_workers=20) as executor:
    futures = {executor.submit(download_one, item): item for item in to_download}
    for i, future in enumerate(as_completed(futures)):
        fname, ok, err = future.result()
        if ok:
            downloaded += 1
        else:
            failures.append((fname, err))
        if (i + 1) % 100 == 0:
            print(f"  Progress: {i+1}/{len(to_download)} "
                  f"(downloaded: {downloaded}, failed: {len(failures)})")

print(f"\nDone: {downloaded} downloaded, {skipped} skipped, {len(failures)} failed")
if failures:
    print("Failures:")
    for fname, err in failures[:20]:
        print(f"  {fname}: {err}")

Total files to download: 1240
Already downloaded: 1240, remaining: 0

Done: 0 downloaded, 1240 skipped, 0 failed


## 2. Parse all files

Five format eras detected from header keywords + data column count:
- **Format A** (2000–2003): 26 cols, MSA/CMSA codes
- **Format B** (2004): 26 cols, CSA/CBSA replaces MSA/CMSA
- **Format B2** (Jan–Mar 2005): 27 cols, Footnote + empty field before name
- **Format B2b** (Apr 2005–Dec 2007): 26 cols, Footnote, no empty field
- **Format C** (2008–2025): 29 cols, FIPS Place, FIPS MCD, Pop added

In [14]:
def detect_format(filepath):
    """Detect CSV format era from header keywords + data column count."""
    with open(filepath, 'r') as f:
        header = f.readline()
        f.readline()  # header line 2
        f.readline()  # blank line
        first_data = f.readline()
    n_cols = len(first_data.split(','))

    if n_cols == 29:
        return 'C'
    elif n_cols == 27:
        return 'B2'     # Jan-Mar 2005: Footnote + empty field
    elif 'Footnote' in header:
        return 'B2b'    # Apr 2005-Dec 2007: Footnote, no empty field
    elif 'CSA' in header or 'CBSA' in header:
        return 'B'
    else:
        return 'A'

FORMAT_COLS = {'A': COLS_A, 'B': COLS_B, 'B2': COLS_B2, 'B2b': COLS_B2b, 'C': COLS_C}


def parse_place_file(filepath):
    """Parse a single BPS place-level file into a DataFrame."""
    fmt = detect_format(filepath)
    col_names = FORMAT_COLS[fmt]

    df = pd.read_csv(
        filepath, skiprows=3, header=None,
        names=col_names, dtype=str,
        on_bad_lines='skip',
    )

    # Drop rows where survey_date is not a valid 6-digit YYYYMM
    df = df[df['survey_date'].str.strip().str.match(r'^\d{6}$', na=False)]

    # Drop internal helper columns
    if '_empty' in df.columns:
        df = df.drop(columns=['_empty'])

    return df, fmt


def standardize_columns(df, fmt):
    """Map format-specific columns to common output columns."""
    out = pd.DataFrame()

    # Columns present in all formats
    for col in ['survey_date', 'state_code', 'id_6digit', 'county_code',
                'place_name', 'zip_code', 'region', 'division', 'source']:
        out[col] = df[col] if col in df.columns else np.nan

    # CSA/CBSA: present in B, B2, C but not A
    out['csa_code'] = df['csa_code'] if 'csa_code' in df.columns else np.nan
    out['cbsa_code'] = df['cbsa_code'] if 'cbsa_code' in df.columns else np.nan

    # Population: only in C
    out['pop'] = df['pop'] if 'pop' in df.columns else np.nan

    # Permit columns (always present)
    for col in PERMIT_COLS:
        out[col] = df[col]

    return out


# Parse all files
all_dfs = []
format_counts = {'A': 0, 'B': 0, 'B2': 0, 'B2b': 0, 'C': 0}
parse_errors = []

files = sorted(RAW_PLACE.glob('*c.txt'))
print(f"Found {len(files)} files to parse")

for i, fp in enumerate(files):
    try:
        df, fmt = parse_place_file(fp)
        df_std = standardize_columns(df, fmt)
        all_dfs.append(df_std)
        format_counts[fmt] += 1
    except Exception as e:
        parse_errors.append((fp.name, str(e)))
    if (i + 1) % 200 == 0:
        print(f"  Parsed {i+1}/{len(files)} files...")

print(f"\nParsed {sum(format_counts.values())} files successfully")
print(f"Format counts: {format_counts}")
if parse_errors:
    print(f"Parse errors ({len(parse_errors)}):")
    for fname, err in parse_errors[:10]:
        print(f"  {fname}: {err}")

# Concatenate once
panel = pd.concat(all_dfs, ignore_index=True)
print(f"\nRaw panel shape: {panel.shape}")

Found 1240 files to parse
  Parsed 200/1240 files...
  Parsed 400/1240 files...
  Parsed 600/1240 files...
  Parsed 800/1240 files...
  Parsed 1000/1240 files...
  Parsed 1200/1240 files...

Parsed 1240 files successfully
Format counts: {'A': 194, 'B': 48, 'B2': 12, 'B2b': 132, 'C': 854}

Raw panel shape: (3278007, 24)


## 3. Clean and transform

In [17]:
# Extract year and month from survey_date (YYYYMM)
panel['survey_date'] = panel['survey_date'].str.strip()
panel['year'] = panel['survey_date'].str[:4].astype(int)
panel['month'] = panel['survey_date'].str[4:6].astype(int)

# Create unique place identifier: state_code (2-digit) + id_6digit (6-digit)
panel['state_code'] = panel['state_code'].str.strip().str.zfill(2)
panel['id_6digit'] = panel['id_6digit'].str.strip()
panel['place_id'] = panel['state_code'] + panel['id_6digit']

# Clean string fields
panel['place_name'] = panel['place_name'].str.strip()
panel['county_code'] = panel['county_code'].str.strip()
panel['zip_code'] = panel['zip_code'].str.strip()
panel['csa_code'] = panel['csa_code'].str.strip().replace('', np.nan)
panel['cbsa_code'] = panel['cbsa_code'].str.strip().replace('', np.nan)

# Convert permit columns to numeric
for col in PERMIT_COLS:
    panel[col] = pd.to_numeric(panel[col], errors='coerce').fillna(0).astype(int)

# Convert pop to numeric (only available 2008+)
panel['pop'] = pd.to_numeric(panel['pop'], errors='coerce')

# Derive region name from file prefix (more reliable than region code)
region_map = {'ne': 'Northeast', 'mw': 'Midwest', 'so': 'South', 'we': 'West'}

print(f"Panel shape: {panel.shape}")
print(f"Date range: {panel['year'].min()}-{panel['month'].min():02d} to "
      f"{panel['year'].max()}-{panel[panel['year']==panel['year'].max()]['month'].max():02d}")
print(f"Unique places: {panel['place_id'].nunique()}")
print(f"\nSample rows:")
panel[['place_id', 'state_code', 'place_name', 'year', 'month',
       'cbsa_code', 'units_1', 'units_5plus']].head(10)

Panel shape: (3278007, 27)
Date range: 2000-01 to 2025-10
Unique places: 21011

Sample rows:


,place_id,state_code,place_name,year,month,cbsa_code,units_1,units_5plus
0,17002800,17,Addison village,2000,1,NaN,1,0
1,17005200,17,Albers village,2000,1,NaN,2,0
2,17007900,17,Algonquin village,2000,1,NaN,16,0
3,17008500,17,Alhambra village,2000,1,NaN,0,0
4,17012100,17,Alpha village,2000,1,NaN,0,0
5,17012700,17,Alsip village,2000,1,NaN,0,0
6,17013000,17,Altamont,2000,1,NaN,0,0
7,17013900,17,Alton,2000,1,NaN,0,44
8,17019300,17,Antioch village,2000,1,NaN,11,0
9,17022300,17,Arlington Heights village,2000,1,NaN,5,0


## 4. Validate

In [20]:
# Check for duplicate place-year-month triples
dupes = panel.groupby(['place_id', 'year', 'month']).size()
dupes_gt1 = dupes[dupes > 1]

if len(dupes_gt1) > 0:
    print(f"WARNING: {len(dupes_gt1)} duplicate place-year-month triples found")
    print(dupes_gt1.head(20))
    # Deduplicate: keep first occurrence
    panel = panel.drop_duplicates(subset=['place_id', 'year', 'month'], keep='first')
    print(f"After dedup: {panel.shape[0]} rows")
else:
    print("No duplicate place-year-month triples. Panel is clean.")

# Month coverage
month_counts = panel.groupby(['year', 'month']).size()
print(f"\nMonths in panel: {len(month_counts)}")
print(f"Places per month (min/median/max): "
      f"{month_counts.min()} / {int(month_counts.median())} / {month_counts.max()}")

# Coverage per place
coverage = panel.groupby('place_id').size()
print(f"\nObservations per place:")
print(coverage.describe())
n_months = panel.groupby(['year', 'month']).ngroups
print(f"\nPlaces with all {n_months} months: {(coverage == n_months).sum()}")
print(f"Places with 200+ months: {(coverage >= 200).sum()}")
print(f"Places with < 50 months: {(coverage < 50).sum()}")

No duplicate place-year-month triples. Panel is clean.

Months in panel: 310
Places per month (min/median/max): 7941 / 9114 / 19994

Observations per place:
count    21011.000000
mean       156.013802
std        110.132702
min          1.000000
25%         46.000000
50%        129.000000
75%        282.000000
max        310.000000
dtype: float64

Places with all 310 months: 2709
Places with 200+ months: 7811
Places with < 50 months: 7975


## 5. Summary statistics

In [23]:
print(f"Total rows: {len(panel):,}")
print(f"Unique places: {panel['place_id'].nunique():,}")
print(f"Unique states: {panel['state_code'].nunique()}")
print(f"Date range: {panel['year'].min()}-{panel['month'].min():02d} to "
      f"{panel['year'].max()}-{panel[panel['year']==panel['year'].max()]['month'].max():02d}")

# Places per state (top 10)
places_per_state = panel.groupby('state_code')['place_id'].nunique().sort_values(ascending=False)
print(f"\nPlaces per state (top 10):")
print(places_per_state.head(10))

# Total permits by type
total_1 = panel['units_1'].sum()
total_2 = panel['units_2'].sum()
total_34 = panel['units_34'].sum()
total_5p = panel['units_5plus'].sum()
total_all = total_1 + total_2 + total_34 + total_5p
print(f"\nTotal units permitted (2000-2025):")
print(f"  1-unit:   {total_1:>12,}  ({100*total_1/total_all:.1f}%)")
print(f"  2-unit:   {total_2:>12,}  ({100*total_2/total_all:.1f}%)")
print(f"  3-4 unit: {total_34:>12,}  ({100*total_34/total_all:.1f}%)")
print(f"  5+ unit:  {total_5p:>12,}  ({100*total_5p/total_all:.1f}%)")
print(f"  Total:    {total_all:>12,}")

# National monthly totals for quick sanity check
national = panel.groupby(['year', 'month'])[PERMIT_COLS].sum()
national['total_units'] = national['units_1'] + national['units_2'] + national['units_34'] + national['units_5plus']
print(f"\nNational monthly total units (range): "
      f"{national['total_units'].min():,} – {national['total_units'].max():,}")

Total rows: 3,278,007
Unique places: 21,011
Unique states: 52
Date range: 2000-01 to 2025-10

Places per state (top 10):
state_code
42    2328
36    1370
55    1194
17    1005
48    1001
27     991
39     969
26     896
19     809
29     586
Name: place_id, dtype: int64

Total units permitted (2000-2025):
  1-unit:     22,092,076  (68.3%)
  2-unit:        592,355  (1.8%)
  3-4 unit:      545,850  (1.7%)
  5+ unit:     9,117,835  (28.2%)
  Total:      32,348,116

National monthly total units (range): 32,727 – 190,873


## 6. Save output

In [26]:
# Sort by place and date
panel = panel.sort_values(['place_id', 'year', 'month']).reset_index(drop=True)

# Select output columns
output_cols = [
    'place_id', 'state_code', 'id_6digit', 'county_code', 'place_name',
    'year', 'month', 'zip_code', 'csa_code', 'cbsa_code', 'pop',
] + PERMIT_COLS

panel[output_cols].to_csv(OUTPUT_DIR / 'bps_place_monthly_panel.csv', index=False)

fsize = (OUTPUT_DIR / 'bps_place_monthly_panel.csv').stat().st_size / 1e6
print(f"Saved to {OUTPUT_DIR / 'bps_place_monthly_panel.csv'}")
print(f"File size: {fsize:.1f} MB")
print(f"Shape: {panel.shape[0]:,} rows × {len(output_cols)} columns")

Saved to C:\Users\Lenovo\bps_dnsc_repo\bps-dnsc-repo\notebooks\output\bps_place_monthly_panel.csv
File size: 313.0 MB
Shape: 3,278,007 rows × 23 columns


## 7. Quick visual check

In [28]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

plt.style.use(str(PROJECT_DIR / 'paper.mplstyle'))
FIGURES_DIR = PROJECT_DIR / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

# National monthly total single-family permits
national = panel.groupby(['year', 'month'])['units_1'].sum().reset_index()
national['date'] = pd.to_datetime(
    national['year'].astype(str) + '-' + national['month'].astype(str) + '-01')
national = national.sort_values('date')

fig, ax = plt.subplots()
ax.plot(national['date'], national['units_1'])
ax.set_ylabel('Single-Family Permits')
# ax.set_title('National Monthly Single-Family Building Permits (Place-Level Aggregation)')

ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_minor_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_ylim(0, 160_000)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x/1000:.0f}k'))

fig.savefig(FIGURES_DIR / 'fig_national_sf_permits.pdf', dpi=150)
plt.show()
print(f"Saved to {FIGURES_DIR / 'fig_national_sf_permits.pdf'}")

OSError: 'C:\\Users\\Lenovo\\bps_dnsc_repo\\bps-dnsc-repo\\notebooks\\paper.mplstyle' is not a valid package style, path of style file, URL of style file, or library style name (library styles are listed in `style.available`)